In [1]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
from plotly import graph_objs

## Consumption Page

In [2]:
df = pd.read_csv("data/sub_rate.csv")
freq = df.sort_values("total_minutes_watched").groupby("buckets", as_index=False, sort=False).student_id.count()
freq["sub_rate"] = df.sort_values("total_minutes_watched").groupby("buckets", sort=False).f2p.sum().values / freq.student_id.values
freq

,buckets,student_id,sub_rate
0,0,18771,0.029993
1,"(0.0, 5.0]",8428,0.023493
2,"(5.0, 10.0]",2044,0.071918
3,"(10.0, 15.0]",1202,0.088186
4,"(15.0, 20.0]",855,0.078363
5,"(20.0, 25.0]",593,0.119730
6,"(25.0, 30.0]",525,0.104762
7,"(30.0, 40.0]",1037,0.154291
8,"(40.0, 50.0]",1164,0.142612
9,"(50.0, 60.0]",579,0.186528


In [3]:
def sub_rate_plot():
    df = pd.read_csv("data/sub_rate.csv")
    freq = df.sort_values("total_minutes_watched").groupby("buckets", as_index=False, sort=False).student_id.count()
    freq["sub_rate"] = df.sort_values("total_minutes_watched").groupby("buckets", sort=False).f2p.sum().values / freq.student_id.values
    
    fig = make_subplots(specs=[[{"secondary_y": True}]], subplot_titles=["Free Plan Students Engagement Groups"])
    fig.add_trace(
        graph_objs.Bar(x=freq.buckets, y=freq.student_id, name="student count"),
        secondary_y=False
    )
    fig.add_trace(
        graph_objs.Scatter(x=freq.buckets, y=freq.sub_rate, name="subscription rate"),
        secondary_y=True
    )
    fig.update_layout(
        hovermode="x unified",
        xaxis=dict(title="minutes watched")
    )

    return fig

fig = sub_rate_plot()
fig.show()

In [4]:
import numpy as np

df = pd.read_csv("data/sub_duration.csv")
freq = df.sort_values("total_minutes_watched").groupby("buckets", as_index=False, sort=False).student_id.count()
df.num_paid_days = df.num_paid_days.str.extract(r"(\d+)").astype(int)
freq["mean_sub_days"] = np.round(df.sort_values("total_minutes_watched").groupby("buckets", sort=False).num_paid_days.sum().values / freq.student_id.values)
freq

,buckets,student_id,mean_sub_days
0,0,164,216.0
1,"(0.0, 30.0]",220,267.0
2,"(30.0, 60.0]",153,280.0
3,"(60.0, 120.0]",239,262.0
4,"(120.0, 240.0]",343,256.0
5,"(240.0, 480.0]",382,267.0
6,"(480.0, 1000.0]",408,265.0
7,"(1000.0, 2000.0]",256,299.0
8,"(2000.0, 3000.0]",84,293.0
9,"(3000.0, 4000.0]",55,327.0


In [5]:
import numpy as np

def sub_duration_plot():
    df = pd.read_csv("data/sub_duration.csv")
    freq = df.sort_values("total_minutes_watched").groupby("buckets", as_index=False, sort=False).student_id.count()
    df.num_paid_days = df.num_paid_days.str.extract(r"(\d+)").astype(int)
    freq["mean_sub_days"] = np.round(df.sort_values("total_minutes_watched").groupby("buckets", sort=False).num_paid_days.sum().values / freq.student_id.values)

    fig = make_subplots(specs=[[{"secondary_y": True}]], subplot_titles=["Subscribed Students Engagement Groups"])
    fig.add_trace(
        graph_objs.Bar(x=freq.buckets, y=freq.student_id, name="student count"),
        secondary_y=False
    )
    fig.add_trace(
        graph_objs.Scatter(x=freq.buckets, y=freq.mean_sub_days, name="Avg. Subscription Duration (days)"),
        secondary_y=True
    )
    fig.update_layout(
        hovermode="x unified",
        xaxis=dict(title="minutes watched")
    )

    return fig

fig = sub_duration_plot()
fig.show()

In [35]:
df = pd.read_csv("data/learning.csv")
df["month"] = pd.to_datetime(df.date_watched).dt.strftime("%Y-%m")
df_grouped = df.groupby(["month", "paid"], as_index=False).minutes_watched.sum()
df_grouped["n_students"] = df.groupby(["month", "paid"]).student_id.nunique().values
df_grouped

,month,paid,minutes_watched,n_students
0,2022-01,0,28211.98,1528
1,2022-01,1,58551.52,255
2,2022-02,0,29325.09,1534
3,2022-02,1,98631.68,359
4,2022-03,0,34909.72,1813
5,2022-03,1,144071.84,501
6,2022-04,0,33444.33,1560
7,2022-04,1,139434.95,547
8,2022-05,0,31844.30,1564
9,2022-05,1,128125.45,588


In [46]:
28212 / 1528

18.463350785340314

In [47]:
58552 / 255

229.6156862745098

In [41]:
df_filtered = df_grouped.groupby("month", as_index=False).minutes_watched.sum()
df_filtered["n_students"] = df_grouped.groupby("month").n_students.sum().values
df_filtered

,month,minutes_watched,n_students
0,2022-01,86763.50,1783
1,2022-02,127956.77,1893
2,2022-03,178981.56,2314
3,2022-04,172879.28,2107
4,2022-05,159969.75,2152
5,2022-06,190328.21,2632
6,2022-07,199230.67,2420
7,2022-08,320939.60,3887
8,2022-09,200908.60,3055
9,2022-10,279416.97,3947


In [51]:
(28212 + 58552) / (1528 + 255)

48.661805945036456

In [104]:
import numpy as np
from typing import Literal

def consumption_plot(stud_type: Literal["All", "Free", "Paid"]):
    df = pd.read_csv("data/learning.csv")
    df["month"] = pd.to_datetime(df.date_watched).dt.strftime("%Y-%m")
    df_grouped = df.groupby(["month", "paid"], as_index=False).minutes_watched.sum()
    df_grouped["n_students"] = df.groupby(["month", "paid"]).student_id.nunique().values
    if stud_type == "All":
        df_filtered = df_grouped.groupby("month", as_index=False).minutes_watched.sum()
        df_filtered["n_students"] = df_grouped.groupby("month").n_students.sum().values
    else:
        paid = {"Free": 0, "Paid": 1}
        df_filtered = df_grouped[df_grouped.paid == paid[stud_type]]

    fig = make_subplots(specs=[[{"secondary_y": True}]], subplot_titles=[f"Total Consumption; Type: {stud_type}"])
    fig.add_trace(
        graph_objs.Bar(x=df_filtered.month, y=df_filtered.minutes_watched, name="Total (min)"),
        secondary_y=False
    )
    mean_mins_per_student = np.round(df_filtered.minutes_watched.values / df_filtered.n_students.values, 2)
    fig.add_trace(
        graph_objs.Scatter(x=df_filtered.month, y=mean_mins_per_student, name="Average (min)"),
        secondary_y=True
    )
    fig.update_layout(
    hovermode="x unified",
    xaxis=dict(title="Month")
    )

    return fig

fig = consumption_plot("Paid")
fig.show()

## Retention Page

In [ ]:
import numpy as np

df = pd.read_csv("data/cohorts.csv")
df_grouped = df.groupby(["cohort", "period"], as_index=False).student_id.nunique()
table = pd.DataFrame(index=np.unique(df_grouped.cohort.values), columns=np.arange(10))
for idx in table.index:
    for col in table.columns:
        try:
            table.loc[idx, col] = df_grouped[(df_grouped.cohort == idx) & (df_grouped.period == col)].student_id.values[0]
        except IndexError:
            continue

table

,0,1,2,3,4,5,6,7,8,9
2022-01,1718,276,164,120,103,97,76,74,71,58
2022-02,1584,273,115,80,78,71,78,58,58,NaN
2022-03,1838,277,162,130,104,103,87,75,NaN,NaN
2022-04,1568,238,115,81,94,63,67,NaN,NaN,NaN
2022-05,1520,242,121,121,93,78,NaN,NaN,NaN,NaN
2022-06,1944,263,207,108,96,NaN,NaN,NaN,NaN,NaN
2022-07,1671,321,170,136,NaN,NaN,NaN,NaN,NaN,NaN
2022-08,2891,327,164,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-09,2038,335,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-10,2848,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [141]:
df_grouped = df.groupby(["cohort", "period"], as_index=False).student_id.nunique()
df_grouped.loc[df_grouped.period == 0, "total"] = df_grouped.student_id
df_grouped.total = df_grouped.total.ffill()
df_grouped["relative"] = np.round(df_grouped.student_id / df_grouped.total * 100, 2)
df_grouped

,cohort,period,student_id,total,relative
0,2022-01,0,1718,1718.0,100.00
1,2022-01,1,276,1718.0,16.07
2,2022-01,2,164,1718.0,9.55
3,2022-01,3,120,1718.0,6.98
4,2022-01,4,103,1718.0,6.00
5,2022-01,5,97,1718.0,5.65
6,2022-01,6,76,1718.0,4.42
7,2022-01,7,74,1718.0,4.31
8,2022-01,8,71,1718.0,4.13
9,2022-01,9,58,1718.0,3.38


In [110]:
def cohort_table(stud_type: Literal["All", "Free", "Paid"]):
    df = pd.read_csv("data/cohorts.csv")
    if stud_type == "All":
        df_grouped = df.groupby(["cohort", "period"], as_index=False).student_id.nunique()
    else:
        paid = {"Free": 0, "Paid": 1}
        df_grouped = df.groupby(["cohort", "period", "paid"], as_index=False).student_id.nunique()
    table = pd.DataFrame(index=np.unique(df_grouped.cohort.values), columns=np.arange(10))
    for idx in table.index:
        for col in table.columns:
            try:
                if stud_type == "All":
                    table.loc[idx, col] = df_grouped[(df_grouped.cohort == idx) & (df_grouped.period == col)].student_id.values[0]
                else:
                    table.loc[idx, col] = df_grouped[(df_grouped.cohort == idx) &
                                                     (df_grouped.period == col) &
                                                     (df_grouped.paid == paid[stud_type])].student_id.values[0]
            except IndexError:
                continue
    
    fig = px.imshow(table, text_auto=True,
                    labels=dict(x="Month", y="Cohort", color="Active Students"),
                    title=f"Retention Cohorts; Type: {stud_type}")
    return fig

fig = cohort_table("All")
fig.show()

In [148]:
def retention_curve(stud_type: Literal["All", "Free", "Paid"]):
    df = pd.read_csv("data/cohorts.csv")
    if stud_type == "All":
        df_grouped = df.groupby(["cohort", "period"], as_index=False).student_id.nunique()
    else:
        paid = {"Free": 0, "Paid": 1}
        df_grouped = df.groupby(["cohort", "period", "paid"], as_index=False).student_id.nunique()
        df_grouped = df_grouped[df_grouped.paid == paid[stud_type]]
    df_grouped.loc[df_grouped.period == 0, "total"] = df_grouped.student_id
    df_grouped.total = df_grouped.total.ffill()
    df_grouped["relative"] = np.round(df_grouped.student_id / df_grouped.total * 100, 2)
    
    fig = px.line(df_grouped, x="period", y="relative", color="cohort",
                  labels={"period": "month", "relative": "percent"})
    return fig

fig = retention_curve("All")
fig.show()

## Engagement Page

In [50]:
import pandas as pd

df = pd.read_csv("data/engagement.csv")
df["month"] = pd.to_datetime(df.engagement_date).dt.strftime("%Y-%m")
df[df.paid == 0]

,student_id,engagement_date,paid,month
0,258815,2022-01-01,0,2022-01
1,258822,2022-01-01,0,2022-01
2,258798,2022-01-01,0,2022-01
3,258807,2022-01-01,0,2022-01
4,258809,2022-01-01,0,2022-01
...,...,...,...,...
68332,297050,2022-10-31,0,2022-10
68333,297052,2022-10-31,0,2022-10
68335,296997,2022-10-31,0,2022-10
68337,297053,2022-10-31,0,2022-10


In [62]:
import numpy as np

np.datetime64("2022-01-01") < np.datetime64("2022-01-02")

np.True_

In [147]:
import plotly.express as px
from typing import Literal
import numpy as np

def engagement_plot(view: Literal["Daily", "Monthly"],
                    stud_type: Literal["All", "Free", "Paid"],
                    period: tuple[np.datetime64]):
    df = pd.read_csv("data/engagement.csv")
    df.engagement_date = pd.to_datetime(df.engagement_date)
    df["month"] = df.engagement_date.dt.strftime("%Y-%m")
    df = df[(df.engagement_date >= period[0]) & (df.engagement_date <= period[1])]

    if stud_type != "All":
        paid = {"Free": 0, "Paid": 1}
        df = df[df.paid == paid[stud_type]]
    
    if view == "Daily":
        df_grouped = df.groupby("engagement_date", as_index=False).student_id.nunique()
        fig = px.line(x=df_grouped.engagement_date, y=df_grouped.student_id,
                      labels=dict(x="Day", y="Active Students"),
                      title=f"{view} Student Engagement; Type: {stud_type}")
    elif view == "Monthly":
        df_grouped = df.groupby("month", as_index=False).student_id.nunique()
        fig = px.bar(x=df_grouped.month, y=df_grouped.student_id,
                     labels=dict(x="Month", y="Active Students"),
                     title=f"{view} Student Engagement; Type: {stud_type}")
    
    return fig

start, end = np.datetime64("2022-01-01"), np.datetime64("2022-10-31")
fig = engagement_plot("Daily", "All", (start, end))
fig.show()

In [32]:
df[df.month == "2022-08"].student_id.unique()

array([282553, 283026, 266477, ..., 283846, 288955, 288957], shape=(3882,))

In [ ]:
df = pd.read_csv("data/onboarded.csv")
df.date_registered = pd.to_datetime(df.date_registered)
df["month"] = df.date_registered.dt.strftime("%Y-%m")
df["monthly_onboard_rate"] = df.groupby("month").onboarded.cumsum() / (df.groupby("month").student_id.cumcount() + 1)
df["overall_onboard_rate"] = df.onboarded.cumsum() / (np.array(df.index) + 1)
df

,student_id,date_registered,onboarded,month,monthly_onboard_rate,overall_onboard_rate
0,258798,2022-01-01,1,2022-01,1.000000,1.000000
1,258799,2022-01-01,0,2022-01,0.500000,0.500000
2,258800,2022-01-01,1,2022-01,0.666667,0.666667
3,258801,2022-01-01,0,2022-01,0.500000,0.500000
4,258802,2022-01-01,0,2022-01,0.400000,0.400000
...,...,...,...,...,...,...
37485,297050,2022-10-31,1,2022-10,0.639257,0.515633
37486,297052,2022-10-31,1,2022-10,0.639344,0.515645
37487,297053,2022-10-31,1,2022-10,0.639431,0.515658
37488,297054,2022-10-31,0,2022-10,0.639277,0.515645


In [158]:
def onboarding_plot(view: Literal["Daily", "Monthly"],
                    period: tuple[np.datetime64]):
    df = pd.read_csv("data/onboarded.csv")
    df.date_registered = pd.to_datetime(df.date_registered)
    df["month"] = df.date_registered.dt.strftime("%Y-%m")
    df["monthly_onboard_rate"] = df.groupby("month").onboarded.cumsum() / (df.groupby("month").student_id.cumcount() + 1)
    df["overall_onboard_rate"] = df.onboarded.cumsum() / (np.array(df.index) + 1)

    df = df[(df.date_registered >= period[0]) & (df.date_registered <= period[1])]

    if view == "Daily":
        fig = px.line(df, x="date_registered", y="overall_onboard_rate",
                      title="Cumulative Onboard Rate")
        fig.update_layout(yaxis_range=[0.4, 0.7])
    elif view == "Monthly":
        fig = px.line(df, x="date_registered", y="monthly_onboard_rate",
                      title="Cumulative Onboard Rate per Month")
        fig.update_layout(yaxis_range=[0.4, 0.7])
    return fig

start, end = np.datetime64("2022-01-01"), np.datetime64("2022-10-31")
fig = onboarding_plot("Daily", (start, end))
fig.show()

## Exams Page

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("data/exam_attempts.csv")
df.date_exam_completed = pd.to_datetime(df.date_exam_completed)
df["month"] = df.date_exam_completed.dt.strftime("%Y-%m")
df_grouped = df.groupby("month", as_index=False).exam_attempt_id.count()
df_grouped

,month,exam_attempt_id
0,2022-01,1841
1,2022-02,2192
2,2022-03,3137
3,2022-04,2724
4,2022-05,2489
5,2022-06,3233
6,2022-07,3573
7,2022-08,7234
8,2022-09,3693
9,2022-10,5243


In [7]:
from plotly import graph_objs
from typing import Literal

def exams_bar_plot(category: Literal["All", "Practice", "Course", "Career Track"]):
    df = pd.read_csv("data/exam_attempts.csv")
    df.date_exam_completed = pd.to_datetime(df.date_exam_completed)
    df["month"] = df.date_exam_completed.dt.strftime("%Y-%m")
    if category != "All":
        cat = {"Practice": 1, "Course": 2, "Career Track": 3}
        df = df[df.exam_category == cat[category]]

    df_fail = df[df.exam_passed == 0].groupby("month", as_index=False).exam_attempt_id.count()
    df_pass = df[df.exam_passed == 1].groupby("month", as_index=False).exam_attempt_id.count()

    fig = graph_objs.Figure(
        data=[
            graph_objs.Bar(name="Exams passed", x=df_pass.month, y=df_pass.exam_attempt_id),
            graph_objs.Bar(name="Exams failed", x=df_fail.month, y=df_fail.exam_attempt_id)
        ]
    )
    fig.update_layout(barmode="stack", title="Exams per Month")
    return fig

fig = exams_bar_plot("All")
fig.show()

In [3]:
df = pd.read_csv("data/career_track_stats.csv")
df

,action,track_id,n_students
0,track enrollments,1,2999
1,track enrollments,2,3915
2,track enrollments,3,960
3,course exam attempts,1,383
4,course exam attempts,2,317
5,course exam attempts,3,163
6,course exam passed,1,344
7,course exam passed,2,299
8,course exam passed,3,148
9,final exam attempts,1,24


In [4]:
def career_track_funnel(track: Literal["Data Scientist", "Data Analyst", "Business Analyst"]):
    df = pd.read_csv("data/career_track_stats.csv")
    track_id = {"Data Scientist": 1, "Data Analyst": 2, "Business Analyst": 3}
    df = df[df.track_id == track_id[track]]
    fig = px.funnel(df, x="n_students", y="action", title=f"{track} Career Track Funnel")
    return fig

fig = career_track_funnel("Data Scientist")
fig.show()

In [13]:
df = pd.read_csv("data/certs.csv")
df.date_issued = pd.to_datetime(df.date_issued)
df["month"] = df.date_issued.dt.strftime("%Y-%m")
df

,cert_id,student_id,cert_type,date_issued,paid,month
0,68911,259209,1,2022-01-04,1,2022-01
1,68946,259258,1,2022-01-05,1,2022-01
2,68969,258955,1,2022-01-06,1,2022-01
3,68976,259268,1,2022-01-06,1,2022-01
4,68990,259447,1,2022-01-07,1,2022-01
...,...,...,...,...,...,...
3678,81557,296993,1,2022-10-31,0,2022-10
3679,81559,272510,1,2022-10-31,1,2022-10
3680,81562,279562,1,2022-10-31,1,2022-10
3681,81563,261682,2,2022-10-31,1,2022-10


In [19]:
def certs_bar_plot(category: Literal["All", "Course", "Career Track"]):
    df = pd.read_csv("data/certs.csv")
    df.date_issued = pd.to_datetime(df.date_issued)
    df["month"] = df.date_issued.dt.strftime("%Y-%m")
    if category != "All":
        cat = {"Course": 1, "Career Track": 2}
        df = df[df.cert_type == cat[category]]

    df_free = df[df.paid == 0].groupby("month", as_index=False).cert_id.count()
    df_paid = df[df.paid == 1].groupby("month", as_index=False).cert_id.count()

    fig = graph_objs.Figure(
        data=[
            graph_objs.Bar(name="Free Plan", x=df_free.month, y=df_free.cert_id),
            graph_objs.Bar(name="Paid Plan", x=df_paid.month, y=df_paid.cert_id)
        ]
    )
    fig.update_layout(barmode="stack", title=f"{category} Certificates Issued")
    return fig

fig = certs_bar_plot("All")
fig.show()

## Home Page

In [39]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("data/ratings.csv")
freq = pd.DataFrame(data=[], columns=["rating", "rating_count"])
freq.rating_count = df.value_counts().values
freq.rating = [f"{i[0]} Star" for i in df.value_counts().index]
freq

,rating,rating_count
0,5 Star,2289
1,4 Star,353
2,3 Star,64
3,2 Star,20
4,1 Star,5


In [44]:
df.course_rating.mean()

np.float64(4.794580739655804)

In [49]:
def ratings_plot():
    df = pd.read_csv("data/ratings.csv")
    freq = pd.DataFrame(data=[], columns=["rating", "rating_count"])
    freq.rating_count = df.value_counts().values
    freq.rating = [f"{i[0]} Star" for i in df.value_counts().index]

    fig = px.pie(freq, values="rating_count", names="rating", hole=0.6,
                 title="Course Ratings")
    fig.add_annotation(text=f"Average<br>Rating:<br>{df.course_rating.mean():.2f}",
                       showarrow=False)

    return fig

fig = ratings_plot()
fig.show()

In [51]:
df = pd.read_csv("data/course_stats.csv")
df

,course_id,title,length_mins,total_mins_watched,mean_mins_per_student,mean_completion_rate
0,2,Introduction to Tableau,95.93,26179.02,40.21,0.42
1,3,The Complete Data Visualization Course with Py...,514.23,44950.58,74.79,0.15
2,4,Introduction to R Programming,386.65,20737.10,69.12,0.18
3,5,Data Preprocessing with NumPy,392.99,37058.31,99.35,0.25
4,7,Introduction to Data and Data Science,110.69,297197.71,29.87,0.27
5,11,Data Cleaning and Preprocessing with pandas,145.20,21701.18,45.12,0.31
6,12,Introduction to Business Analytics,283.87,10002.14,38.62,0.14
7,13,Data Analysis with Excel Pivot Tables,54.47,22321.72,23.08,0.42
8,14,SQL,491.35,215894.11,114.72,0.23
9,15,Credit Risk Modeling in Python,388.20,19513.04,42.98,0.11


In [ ]:
from typing import Literal

def courses_plot(metric: Literal["Total", "Per Student", "Completion Rate"],
                 limit: int):
    metrics = {"Total": "total_mins_watched",
               "Per Student": "mean_mins_per_student",
               "Completion Rate": "mean_completion_rate"}
    df = pd.read_csv("data/course_stats.csv").sort_values(metrics[metric])[-limit:]

    unit = " (min)" if metric != "Completion Rate" else ""

    fig = px.bar(df, y="title", x=metrics[metric], orientation="h",
                 title=f"Most Consumed Courses ({metric})")
    fig.update_layout(xaxis=dict(title=metric + unit))

    return fig

fig = courses_plot("Total", 10)
fig.show()

In [109]:
df = pd.read_csv("data/engagement.csv")
df

,student_id,engagement_date,paid
0,258815,2022-01-01,0
1,258822,2022-01-01,0
2,258798,2022-01-01,0
3,258807,2022-01-01,0
4,258809,2022-01-01,0
...,...,...,...
68335,296997,2022-10-31,0
68336,291116,2022-10-31,1
68337,297053,2022-10-31,0
68338,271950,2022-10-31,1


In [137]:
from plotly import graph_objs
import numpy as np

def engagement_kpi(period: tuple[np.datetime64],
                   stud_type: Literal["All", "Free", "Paid"]):
    df = pd.read_csv("data/engagement.csv")
    df.engagement_date = pd.to_datetime(df.engagement_date)
    df = df[(df.engagement_date >= period[0]) & (df.engagement_date <= period[1])]
    if stud_type != "All":
        paid = {"Free": 0, "Paid": 1}
        df = df[df.paid == paid[stud_type]]
    n = df.student_id.nunique()

    fig = graph_objs.Figure(
        data=graph_objs.Indicator(
            mode="number",
            value=n,
            title=dict(text="Engaged Students")
        )
    )
    
    return fig

start, end = np.datetime64("2022-01-01"), np.datetime64("2022-10-31")
fig = engagement_kpi((start, end), "All")
fig.show()

In [118]:
df = pd.read_csv("data/learning.csv")
df

,student_id,date_watched,minutes_watched,paid
0,258940,2022-01-02,3.52,0
1,258893,2022-01-02,25.25,0
2,258904,2022-01-02,48.11,0
3,259041,2022-01-02,0.18,0
4,258904,2022-01-03,82.56,0
...,...,...,...,...
67431,291497,2022-10-30,6.33,0
67432,264898,2022-10-30,1.48,0
67433,271154,2022-10-30,39.00,1
67434,293600,2022-10-31,159.53,1


In [ ]:
def time_watched_kpi(period: tuple[np.datetime64],
                     stud_type: Literal["All", "Free", "Paid"]):
    df = pd.read_csv("data/learning.csv")
    df.date_watched = pd.to_datetime(df.date_watched)
    df = df[(df.date_watched >= period[0]) & (df.date_watched <= period[1])]
    if stud_type != "All":
        paid = {"Free": 0, "Paid": 1}
        df = df[df.paid == paid[stud_type]]
    n = df.minutes_watched.sum() / df.student_id.nunique()

    fig = graph_objs.Figure(
        data=graph_objs.Indicator(
            mode="number",
            value=n,
            title=dict(text="Mean Minutes Watched per Student")
        )
    )
    
    return fig

start, end = np.datetime64("2022-01-01"), np.datetime64("2022-10-31")
fig = time_watched_kpi((start, end), "All")
fig.show()

In [140]:
df = pd.read_csv("data/certs.csv")
df

,cert_id,student_id,cert_type,date_issued,paid
0,68911,259209,1,2022-01-04,1
1,68946,259258,1,2022-01-05,1
2,68969,258955,1,2022-01-06,1
3,68976,259268,1,2022-01-06,1
4,68990,259447,1,2022-01-07,1
...,...,...,...,...,...
3678,81557,296993,1,2022-10-31,0
3679,81559,272510,1,2022-10-31,1
3680,81562,279562,1,2022-10-31,1
3681,81563,261682,2,2022-10-31,1


In [142]:
df.cert_id.count()

np.int64(3683)

In [174]:
def certs_kpi(period: tuple[np.datetime64],
              stud_type: Literal["All", "Free", "Paid"]):
    df = pd.read_csv("data/certs.csv")
    df.date_issued = pd.to_datetime(df.date_issued)
    df = df[(df.date_issued >= period[0]) & (df.date_issued <= period[1])]
    if stud_type != "All":
        paid = {"Free": 0, "Paid": 1}
        df = df[df.paid == paid[stud_type]]
    n = df.cert_id.count()

    fig = graph_objs.Figure(
        data=graph_objs.Indicator(
            mode="number",
            value=n,
            title=dict(text="Certificates Issued"),
            number=dict(valueformat="f")
        )
    )
    
    return fig

start, end = np.datetime64("2022-01-01"), np.datetime64("2022-10-31")
fig = certs_kpi((start, end), "All")
fig.show()